# MLflow, end to end — interactive notebook

A hands-on companion to this project's `README.md`. We walk through nine
progressive lessons that take you from your very first `mlflow.log_param`
call to a full Model Registry workflow with hyperparameter tuning, custom
PyFunc models, and reproducible MLflow Projects.

The dataset is **California Housing** (regression), shipped with
scikit-learn — nothing has to be downloaded.

> **Lesson 10** (production tracking server with Docker Compose) is not
> in this notebook because it requires Docker and a separate server
> process. See the project `README.md` if you want to walk through it.

---

## What you'll cover

| # | Topic | Key MLflow APIs |
|---|---|---|
| 1 | Your first run | `set_experiment`, `start_run`, `log_param`, `log_metric`, `log_artifact`, `log_model` |
| 2 | Experiments, tags, descriptions | `set_tag`, `mlflow.note.content`, `search_runs` |
| 3 | Comparing model families | one run per estimator in one experiment |
| 4 | Autologging | `mlflow.sklearn.autolog`, `mlflow.xgboost.autolog` |
| 5 | Signatures and loading | `infer_signature`, `mlflow.pyfunc.load_model` |
| 6 | Hyperparameter tuning | nested runs (`start_run(nested=True)`) with Optuna |
| 7 | Model Registry | `register_model`, aliases (`@champion`, `@challenger`) |
| 8 | Custom PyFunc models | `mlflow.pyfunc.PythonModel`, `pyfunc.log_model` |
| 9 | MLflow Projects | `mlflow.projects.run` |

## 0. Setup

The notebook reuses the helpers from `src/utils.py` so it stays in sync
with the standalone scripts. We also set the tracking URI explicitly so
it works regardless of where Jupyter was launched from.

In [ ]:
import sys
from pathlib import Path

# Find the project root whether the notebook is launched from notebooks/ or from the repo root.
HERE = Path.cwd()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks" else HERE
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root :", PROJECT_ROOT)

In [ ]:
import mlflow
from src.utils import load_data, regression_metrics

TRACKING_URI = f"file:{PROJECT_ROOT / 'mlruns'}"
mlflow.set_tracking_uri(TRACKING_URI)
print("Tracking URI :", mlflow.get_tracking_uri())

In [ ]:
data = load_data()
print(f"Train: {data.X_train.shape}   Test: {data.X_test.shape}")
data.X_train.head()

## Mental model: the four pillars of MLflow

Before running anything, here is what MLflow actually is. It is four
loosely-coupled components — you can use any subset:

1. **Tracking** — a server (or a local folder) that stores **runs**.
   Each run has params, metrics, tags, and arbitrary file artifacts.
2. **Models** — a standard package format for trained models, with optional
   **signatures** (input/output schemas) and **input examples**.
3. **Model Registry** — gives models a stable name, versions, and aliases
   such as `@champion` / `@challenger`.
4. **Projects** — a folder with an `MLproject` file that turns a script
   into a reproducible, parameterised job.

A useful one-liner: **runs are facts, registered models are decisions**.
You log everything to runs; you promote the runs you trust into the
registry.

## Lesson 1 — Your first MLflow run

The minimal MLflow call site has just five moving parts:

```python
mlflow.set_experiment("lesson-01-first-run")     # bucket of related runs
with mlflow.start_run(run_name="my-run"):        # the run is open here
    mlflow.log_param("alpha", 0.1)               # one-shot, immutable
    mlflow.log_metric("rmse", 0.42)              # numeric, can be updated
    mlflow.log_artifact("plot.png")              # any file
    mlflow.sklearn.log_model(model, "model")     # the model itself
```

We train a Linear Regression baseline, log a residuals plot as an artifact,
and log the model so it can be loaded back later.

In [ ]:
import matplotlib.pyplot as plt
import mlflow.sklearn
from sklearn.linear_model import LinearRegression

from src.utils import ensure_dir

mlflow.set_experiment("lesson-01-first-run")
artifacts_dir = ensure_dir(PROJECT_ROOT / "outputs" / "lesson_01")

with mlflow.start_run(run_name="linear-regression-baseline") as run:
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("n_features", data.X_train.shape[1])

    model = LinearRegression()
    model.fit(data.X_train, data.y_train)
    preds = model.predict(data.X_test)

    metrics = regression_metrics(data.y_test, preds)
    mlflow.log_metrics(metrics)

    # Residuals plot as a file artifact.
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(preds, data.y_test - preds, alpha=0.3, s=10)
    ax.axhline(0, color="red", linewidth=1)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Residual")
    ax.set_title("Residuals — Linear Regression")
    fig.tight_layout()
    plot_path = artifacts_dir / "residuals.png"
    fig.savefig(plot_path, dpi=120); plt.close(fig)
    mlflow.log_artifact(str(plot_path), artifact_path="plots")

    mlflow.sklearn.log_model(sk_model=model, artifact_path="model")
    LESSON1_RUN_ID = run.info.run_id

print("Run id :", LESSON1_RUN_ID)
print("Metrics:", metrics)

### Launching the UI

In a terminal (from the project root):

```bash
mlflow ui --backend-store-uri ./mlruns
```

Then open http://127.0.0.1:5000 and look for the experiment
`lesson-01-first-run`.

You should see:
- one run in the table,
- **Parameters**, **Metrics**, **Artifacts** (residuals plot under
  `plots/`), and a serialized model under `model/`,
- the run is `FINISHED` because the `with` block exited cleanly.

## Lesson 2 — Experiments, runs, tags, descriptions

A real project produces many experiments and many runs per experiment.
Tags and descriptions are how you keep them findable months later.

Three things to learn here:

```python
# Tags on the experiment itself — "what is this experiment about?"
client.create_experiment(name=..., tags={"owner": "george", ...})

# Tags on a run — short, searchable, free-form strings
mlflow.set_tag("model_family", "linear")

# A run *description* is just a special tag (renders as Markdown in the UI)
mlflow.set_tag("mlflow.note.content", "Ridge sweep over alpha = ...")
```

In [ ]:
import platform
from datetime import datetime, timezone
from sklearn.linear_model import Ridge

client = mlflow.MlflowClient()

EXPERIMENT_NAME = "lesson-02-ridge-sweep"
existing = client.get_experiment_by_name(EXPERIMENT_NAME)
if existing is None:
    exp_id = client.create_experiment(
        name=EXPERIMENT_NAME,
        tags={
            "owner": "george",
            "problem": "regression",
            "dataset": "california-housing",
            "stage": "exploration",
            "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        },
    )
else:
    exp_id = existing.experiment_id

print("Experiment id:", exp_id)

In [ ]:
for alpha in [0.01, 0.1, 1.0, 10.0]:
    with mlflow.start_run(experiment_id=exp_id, run_name=f"ridge-alpha-{alpha}"):
        mlflow.set_tag("model_family", "linear")
        mlflow.set_tag("estimator", "Ridge")
        mlflow.set_tag("python_version", platform.python_version())
        mlflow.set_tag(
            "mlflow.note.content",
            f"Ridge regression with `alpha={alpha}`. "
            "Sweep aims to find the best regularisation strength.",
        )

        mlflow.log_params({"alpha": alpha, "fit_intercept": True})

        model = Ridge(alpha=alpha, random_state=0).fit(data.X_train, data.y_train)
        preds = model.predict(data.X_test)
        mlflow.log_metrics(regression_metrics(data.y_test, preds))

print("Done — four runs logged.")

### Searching runs programmatically

Equivalent to the search bar in the UI — useful for CI and notebooks.

In [ ]:
top = mlflow.search_runs(
    experiment_ids=[exp_id],
    order_by=["metrics.rmse ASC"],
    max_results=5,
)
top[["run_id", "params.alpha", "metrics.rmse", "metrics.r2"]]

## Lesson 3 — Comparing multiple model families

Now we move past the linear baseline and train three different estimators
in the same experiment so the UI can compare them side by side.

A small trick: for tree-based models we log every `feature_importances_`
value as its own metric. The UI's **Compare** view then plots them
together for free.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor

mlflow.set_experiment("lesson-03-model-comparison")

candidates = [
    ("ridge",  Ridge(alpha=1.0, random_state=0), {"alpha": 1.0}),
    ("random-forest",
     RandomForestRegressor(n_estimators=200, max_depth=12, n_jobs=-1, random_state=0),
     {"n_estimators": 200, "max_depth": 12}),
    ("gradient-boosting",
     GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=0),
     {"n_estimators": 300, "learning_rate": 0.05, "max_depth": 3}),
]

for name, estimator, params in candidates:
    with mlflow.start_run(run_name=name):
        mlflow.set_tag("model_family", "linear" if "ridge" in name else "tree")
        mlflow.log_param("estimator", estimator.__class__.__name__)
        mlflow.log_params(params)

        estimator.fit(data.X_train, data.y_train)
        preds = estimator.predict(data.X_test)
        mlflow.log_metrics(regression_metrics(data.y_test, preds))

        if hasattr(estimator, "feature_importances_"):
            for feat, imp in zip(data.feature_names, estimator.feature_importances_):
                mlflow.log_metric(f"feature_importance__{feat}", float(imp))

        mlflow.sklearn.log_model(sk_model=estimator, artifact_path="model")

mlflow.search_runs(
    experiment_names=["lesson-03-model-comparison"],
    order_by=["metrics.rmse ASC"],
)[["tags.mlflow.runName", "params.estimator", "metrics.rmse", "metrics.r2"]]

## Lesson 4 — Autologging

For supported libraries you don't need to call `log_param` / `log_metric`
yourself at all. One line at the top of the script and every `model.fit`
inside an active run produces a fully populated MLflow run.

Two gotchas:
- **Autologgers persist** for the whole Python process. If you switch
  libraries, call `mlflow.sklearn.autolog(disable=True)` first.
- Autologgers don't replace your own logging — you can still call
  `log_metric("custom_kpi", ...)` to add domain metrics on top.

In [ ]:
mlflow.set_experiment("lesson-04-autologging")
mlflow.sklearn.autolog(log_input_examples=True, log_model_signatures=True)

with mlflow.start_run(run_name="rf-autolog"):
    rf = RandomForestRegressor(n_estimators=150, max_depth=10, n_jobs=-1, random_state=0)
    rf.fit(data.X_train, data.y_train)
    test_rmse = regression_metrics(data.y_test, rf.predict(data.X_test))["rmse"]
    mlflow.log_metric("test_rmse", test_rmse)
    print(f"Random Forest test RMSE: {test_rmse:.4f}")

mlflow.sklearn.autolog(disable=True)

In [ ]:
import xgboost as xgb

mlflow.xgboost.autolog(log_input_examples=True, log_model_signatures=True)

dtrain = xgb.DMatrix(data.X_train, label=data.y_train)
dtest  = xgb.DMatrix(data.X_test,  label=data.y_test)
params = {
    "objective": "reg:squarederror",
    "eta": 0.05, "max_depth": 6,
    "subsample": 0.8, "colsample_bytree": 0.8,
    "eval_metric": "rmse",
}

with mlflow.start_run(run_name="xgb-autolog"):
    booster = xgb.train(
        params, dtrain, num_boost_round=400,
        evals=[(dtrain, "train"), (dtest, "valid")],
        verbose_eval=False,
    )
    final_rmse = regression_metrics(data.y_test, booster.predict(dtest))["rmse"]
    mlflow.log_metric("final_test_rmse", final_rmse)
    print(f"XGBoost final test RMSE: {final_rmse:.4f}")

mlflow.xgboost.autolog(disable=True)

With `mlflow.xgboost.autolog()` you also get **per-iteration** training
and validation metrics — those become a beautiful loss curve in the UI's
**Metrics → Chart** view for the `xgb-autolog` run.

## Lesson 5 — Signatures, input examples, and loading models

A **signature** is the typed input/output schema of a model. It's stored
in the `MLmodel` file and powers schema validation when the model is
served. An **input example** is a small DataFrame stored next to the
model so reviewers can see what the model expects.

There are two ways to load a logged model:
- **Native** (`mlflow.sklearn.load_model`) → original sklearn estimator.
- **Generic** (`mlflow.pyfunc.load_model`) → uniform `.predict()` interface
  used by serving and by downstream code that doesn't care which library
  produced the model.

In [ ]:
from mlflow.models import infer_signature

mlflow.set_experiment("lesson-05-signatures")

model = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=3, random_state=0
).fit(data.X_train, data.y_train)

signature = infer_signature(data.X_train, model.predict(data.X_train))
input_example = data.X_train.head(3)

with mlflow.start_run(run_name="gbr-with-signature") as run:
    mlflow.log_params(model.get_params())
    mlflow.log_metrics(regression_metrics(data.y_test, model.predict(data.X_test)))
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=input_example,
    )
    LESSON5_RUN_ID = run.info.run_id

print("Run id:", LESSON5_RUN_ID)

In [ ]:
model_uri = f"runs:/{LESSON5_RUN_ID}/model"
sk_model     = mlflow.sklearn.load_model(model_uri)
pyfunc_model = mlflow.pyfunc.load_model(model_uri)

sample = data.X_test.head(5)
print("sklearn flavour :", sk_model.predict(sample).round(3).tolist())
print("pyfunc flavour  :", pyfunc_model.predict(sample).round(3).tolist())

In the UI, open the run and click the `model/` artifact. Notice the new
**Schema** tab and the saved input example. Both come from the
`signature` and `input_example` arguments above.

## Lesson 6 — Hyperparameter tuning with nested runs

When you run a 25-trial Optuna study you don't want 25 sibling runs
cluttering the experiment. MLflow has **nested runs** for exactly this:

```python
with mlflow.start_run(run_name="optuna-study") as parent:
    def objective(trial):
        with mlflow.start_run(nested=True, run_name=f"trial-{trial.number}"):
            ...
    study.optimize(objective, n_trials=25)
```

The UI then collapses children under the parent. We also refit a final
model with the best params and log it on the **parent** run, so there is
one canonical artifact for the whole study.

In [ ]:
import logging
import optuna

mlflow.set_experiment("lesson-06-tuning")
optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger("mlflow").setLevel(logging.WARNING)

N_TRIALS = 15  # smaller in a notebook to keep runtime modest

with mlflow.start_run(run_name="optuna-gbr-study") as parent:
    mlflow.set_tag("search_method", "optuna-tpe")
    mlflow.log_param("n_trials", N_TRIALS)

    def objective(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 100, 500),
            "learning_rate":    trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "max_depth":        trial.suggest_int("max_depth", 2, 6),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        }
        with mlflow.start_run(nested=True, run_name=f"trial-{trial.number:03d}"):
            mlflow.log_params(params)
            m = GradientBoostingRegressor(random_state=0, **params).fit(data.X_train, data.y_train)
            metrics = regression_metrics(data.y_test, m.predict(data.X_test))
            mlflow.log_metrics(metrics)
            return metrics["rmse"]

    study = optuna.create_study(direction="minimize", study_name="gbr-rmse")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

    best = study.best_trial
    mlflow.log_metric("best_rmse", best.value)
    mlflow.log_params({f"best_{k}": v for k, v in best.params.items()})

    final = GradientBoostingRegressor(random_state=0, **best.params).fit(data.X_train, data.y_train)
    mlflow.sklearn.log_model(
        sk_model=final,
        artifact_path="best_model",
        signature=infer_signature(data.X_train, final.predict(data.X_train)),
        input_example=data.X_train.head(3),
    )
    PARENT_RUN_ID = parent.info.run_id

print(f"Best trial #{best.number}  RMSE={best.value:.4f}")
print("Best params:", best.params)

In the UI, open the parent run and expand the **child runs** section to
see all 15 trials. The parent has its own `best_model/` artifact and
`best_*` parameter entries — that's what should be promoted to the
registry, not an individual trial.

## Lesson 7 — The Model Registry

So far our models have been identified by run id. That doesn't scale.
The registry adds a **stable name** and **versions**:

| URI form | What it points at |
|---|---|
| `runs:/8f0c.../model` | unstable — who has `8f0c` memorised? |
| `models:/<name>/2` | stable: version 2 of a named model |
| `models:/<name>@champion` | even better: a movable alias |

> **Stages vs. aliases.** The old `Staging` / `Production` / `Archived`
> stages are deprecated in favour of aliases — arbitrary strings like
> `champion`, `eu-prod`, `ab-test-week-12`. This tutorial uses aliases.

In [ ]:
REGISTERED_MODEL = "california-housing-regressor"
mlflow.set_experiment("lesson-07-registry")

def _train_and_log(run_name, estimator):
    with mlflow.start_run(run_name=run_name) as run:
        estimator.fit(data.X_train, data.y_train)
        preds = estimator.predict(data.X_test)
        metrics = regression_metrics(data.y_test, preds)
        mlflow.log_metrics(metrics)
        mlflow.log_params(estimator.get_params())
        mlflow.sklearn.log_model(
            sk_model=estimator,
            artifact_path="model",
            signature=infer_signature(data.X_train, estimator.predict(data.X_train)),
            input_example=data.X_train.head(3),
        )
        return run.info.run_id, metrics["rmse"]

rf_run, rf_rmse   = _train_and_log("rf-candidate",
                                   RandomForestRegressor(n_estimators=200, max_depth=12,
                                                         n_jobs=-1, random_state=0))
gbr_run, gbr_rmse = _train_and_log("gbr-candidate",
                                   GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                             max_depth=3, random_state=0))
winner_run = gbr_run if gbr_rmse < rf_rmse else rf_run
print(f"Winner run: {winner_run}  (RF={rf_rmse:.4f}  GBR={gbr_rmse:.4f})")

In [ ]:
# Create the registered model if needed.
try:
    client.get_registered_model(REGISTERED_MODEL)
except Exception:
    client.create_registered_model(name=REGISTERED_MODEL,
                                   description="California housing regressor.")

# Register the winner as version 1, alias = champion.
v1 = mlflow.register_model(f"runs:/{winner_run}/model", REGISTERED_MODEL)
client.update_model_version(REGISTERED_MODEL, v1.version,
                            description="Initial production candidate.")
client.set_registered_model_alias(REGISTERED_MODEL, alias="champion", version=v1.version)
print(f"Registered v{v1.version}  alias=champion")

In [ ]:
# Train an "improved" candidate and register it as v2 / challenger.
challenger_run, _ = _train_and_log(
    "gbr-challenger",
    GradientBoostingRegressor(n_estimators=500, learning_rate=0.03, max_depth=4, random_state=0),
)
v2 = mlflow.register_model(f"runs:/{challenger_run}/model", REGISTERED_MODEL)
client.update_model_version(REGISTERED_MODEL, v2.version,
                            description="Deeper GBR with smaller learning rate.")
client.set_registered_model_alias(REGISTERED_MODEL, alias="challenger", version=v2.version)
print(f"Registered v{v2.version}  alias=challenger")

In [ ]:
# Downstream code stays version-agnostic — it just loads by alias.
champion   = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL}@champion")
challenger = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL}@challenger")

sample = data.X_test.head(5)
print("champion  :", champion.predict(sample).round(3).tolist())
print("challenger:", challenger.predict(sample).round(3).tolist())

In the UI, open the **Models** tab in the top nav. You'll see the
registered model, its versions, and the aliases attached to each.
Promotion is just *moving the alias*: when challenger wins in production,
point `champion` at v2 and your serving code instantly switches.

## Lesson 8 — Custom PyFunc models

A real model often isn't just `estimator.predict(X)`. You may need:
- input clipping or imputation,
- a learned scaler that has to live with the model,
- post-processing into business categories (`low`/`mid`/`high`),
- or even a small ensemble of two estimators.

The `mlflow.pyfunc.PythonModel` interface lets you wrap *anything* into
a model that MLflow can log, load, register, and serve like any other.

In [ ]:
import mlflow.pyfunc
import numpy as np
import pandas as pd

class PriceBandModel(mlflow.pyfunc.PythonModel):
    """Bundles preprocessing + estimator + post-processing."""
    BAND_EDGES = (1.5, 3.0)  # split predictions into low/mid/high

    def __init__(self, estimator, feature_clips):
        self.estimator = estimator
        self.feature_clips = feature_clips

    def predict(self, context, model_input, params=None):
        X = self._clip(model_input)
        raw = self.estimator.predict(X)
        return pd.DataFrame({"prediction": raw, "band": [self._band(p) for p in raw]})

    def _clip(self, X):
        X = X.copy()
        for col, (lo, hi) in self.feature_clips.items():
            if col in X.columns:
                X[col] = X[col].clip(lower=lo, upper=hi)
        return X

    @classmethod
    def _band(cls, value):
        lo, hi = cls.BAND_EDGES
        return "low" if value < lo else "mid" if value < hi else "high"

In [ ]:
mlflow.set_experiment("lesson-08-custom-pyfunc")

estimator = GradientBoostingRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=3, random_state=0
).fit(data.X_train, data.y_train)

# Pick clip ranges from the training distribution to harden against outliers.
feature_clips = {
    col: (float(np.quantile(data.X_train[col], 0.001)),
          float(np.quantile(data.X_train[col], 0.999)))
    for col in data.feature_names
}

pyfunc_model = PriceBandModel(estimator=estimator, feature_clips=feature_clips)

sample_input  = data.X_train.head(5)
sample_output = pyfunc_model.predict(context=None, model_input=sample_input)
signature = infer_signature(sample_input, sample_output)

with mlflow.start_run(run_name="price-band-pyfunc") as run:
    mlflow.log_params({
        "wrapped_estimator": estimator.__class__.__name__,
        "band_edges": str(PriceBandModel.BAND_EDGES),
    })
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=pyfunc_model,
        signature=signature,
        input_example=sample_input,
        pip_requirements=["mlflow", "scikit-learn", "pandas", "numpy"],
    )
    PYFUNC_RUN_ID = run.info.run_id

loaded = mlflow.pyfunc.load_model(f"runs:/{PYFUNC_RUN_ID}/model")
loaded.predict(data.X_test.head(5))

The reloaded model has the same uniform interface as any other —
serving and offline scoring stay identical. This is the pattern most
real teams use, because the artifact in the registry is *the* deployable
thing, not "the estimator plus a doc page explaining how to call it".

## Lesson 9 — MLflow Projects (reproducible runs)

An MLflow Project is just a folder with three things:
- `MLproject` — declares entry points, parameters, and commands.
- `python_env.yaml` (or `conda.yaml`) — pins the environment.
- a script — the actual training code.

Once that exists, anyone can reproduce the run identically with one
command:

```bash
mlflow run mlproject -P n_estimators=500 --env-manager=local
mlflow run https://github.com/<you>/mlflow_tutorial.git#mlproject -P ...
```

Below we trigger the project at `../mlproject` through the Python API so
it's runnable from the notebook. `env_manager="local"` reuses the
current kernel's Python; drop that flag in production and MLflow will
build a fresh virtualenv from `python_env.yaml`.

In [ ]:
project_path = PROJECT_ROOT / "mlproject"
print("Project at:", project_path)
print("Files     :", [p.name for p in project_path.iterdir()])

In [ ]:
submitted = mlflow.projects.run(
    uri=str(project_path),
    entry_point="main",
    parameters={"n_estimators": 400, "learning_rate": 0.04, "max_depth": 4},
    env_manager="local",
    experiment_name="lesson-09-mlproject",
)
print("Status:", submitted.get_status())
print("Run id:", submitted.run_id)

## Wrap up

You now have working code (and tracked runs in `./mlruns`) for:

- the basic tracking API,
- experiment / run organisation,
- multi-model comparison,
- autologging with sklearn and XGBoost,
- model signatures and input examples,
- nested runs for hyperparameter tuning,
- the Model Registry with aliases,
- custom PyFunc models that bundle preprocessing,
- MLflow Projects.

### What's next

- **Lesson 10** in `README.md` brings up a production tracking server in
  Docker Compose (Postgres + MinIO + MLflow). Use the same code you just
  wrote — only the tracking URI changes.
- **Serving** — `mlflow models serve -m "models:/california-housing-regressor@champion"`
  exposes the registered champion as a REST API. The README has the full
  `curl` example.
- **`mlflow.evaluate`** — produces a standardised set of diagnostic plots
  and metrics for any logged model.
- **Datasets API** — `mlflow.data.from_pandas(df).log()` records the
  dataset's hash, schema, and source as part of a run, so reruns can
  detect data drift.

Start the UI from a terminal at the project root and explore everything
you just logged:

```bash
mlflow ui --backend-store-uri ./mlruns
```